# DSv4 Pallas-TPU kernel bench on Colab (v5e-1)

Runs `bench.py` from [`HyperBlaze456/auto-jax-kernel`](https://github.com/HyperBlaze456/auto-jax-kernel) on a single TPU v5e chip.

**Before running:** `Runtime > Change runtime type > TPU` (Colab's TPU runtime is a single **v5e-1** chip, 16 GB HBM). Then `Runtime > Run all`.

The kernels were developed against **jax 0.6.2 + Pallas**, so we pin that exact version — newer JAX drifts the Pallas API and can break the kernels.

In [ ]:
# 1. Install the JAX the kernels were written against, with TPU support.
#    Run this FIRST, before importing jax. If a later cell errors with a
#    libtpu / 'Unable to initialize backend tpu' message, do
#    Runtime > Restart session, then Run all again (skip re-installing).
!pip install -q "jax[tpu]==0.6.2" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
# 2. Confirm we're on a TPU v5e and JAX sees exactly one chip.
import jax
print("jax", jax.__version__, "| jaxlib", jax.lib.__version__)
devs = jax.devices()
print("devices:", devs)
assert devs, "No JAX devices — did you pick the TPU runtime? Runtime > Change runtime type > TPU"
kind = getattr(devs[0], "device_kind", "")
print("device_kind:", kind)
assert "v5e" in kind.lower() or "tpu" in kind.lower(), f"Expected a TPU, got {kind!r}"

In [ ]:
# 3. Get the code. Clones the latest dsv4-kernel branch (idempotent: re-run pulls).
#    NOTE: this fetches whatever is pushed to origin — push your local commits first.
import os
REPO = "https://github.com/HyperBlaze456/auto-jax-kernel.git"
BRANCH = "dsv4-kernel"
if not os.path.isdir("/content/auto-jax-kernel"):
    !git clone -b {BRANCH} {REPO} /content/auto-jax-kernel
else:
    !git -C /content/auto-jax-kernel fetch --quiet origin && git -C /content/auto-jax-kernel checkout {BRANCH} && git -C /content/auto-jax-kernel pull --ff-only
%cd /content/auto-jax-kernel
!git log --oneline -3

### Alt: upload instead of clone
If you'd rather not push (private WIP), skip cell 3 and instead run `tar czf dsv4.tgz dsv4 bench.py` locally, then in a new cell:
```python
from google.colab import files; files.upload()   # pick dsv4.tgz
!mkdir -p /content/auto-jax-kernel && tar xzf dsv4.tgz -C /content/auto-jax-kernel
%cd /content/auto-jax-kernel
```

In [ ]:
# 4. Correctness sanity (tiny shape). Expect `status: pass`.
!python bench.py --preset small_csa --seq 64
!python bench.py --preset small_hca --seq 64

In [ ]:
# 5. Forward benchmarks. peak_tflops auto-detects to 197 (TPU v5e) -> MFU is meaningful.
#    Starting conservative for 16 GB HBM; scale seq up until you hit OOM.
!python bench.py --preset dsv4_flash_csa --seq 4096
!python bench.py --preset dsv4_flash_hca --seq 2048
# Scale-up (uncomment as the chip allows; the eager ref materializes full K/logits):
# !python bench.py --preset dsv4_flash_csa --seq 8192
# !python bench.py --preset dsv4_flash_csa --seq 16384
# !python bench.py --preset dsv4_flash_hca --seq 16384

In [ ]:
# 6. (optional) Forward + backward timing.
# !python bench.py --preset dsv4_flash_csa --seq 4096 --bwd

In [ ]:
# 7. (optional) Full sweep across presets x a seq ladder.
#    On a single v5e-1 the dsv4_pro_* presets (d=7168, n_h=128) and seq>=65536
#    will likely OOM; the sweep logs those as ERROR and continues.
# !python bench.py --sweep